# README

the comments (or rows) starting with ">" show Windows Command Prompts  
necessary for the execution.  

1. create and activate a python virtual environment
```> python -m venv .venv```
```> .\.venv\Scripts\activate```

2. install pandas. you'll need it to clients
```> python -m pip install pandas```

In [14]:
# import clients and select some for code

# add path for flie clients.csv
# import sys
# from pathlib import Path

# project_root = Path(__file__).resolve().parents[1]
# classes_path = project_root / "data"
# sys.path.insert(0, str(classes_path))

# create dataframe
import pandas as pd

pd.options.display.expand_frame_repr = False

pd.options.display.width = None
pd.options.display.max_columns = None

clients_pd = pd.read_csv("data/clients.csv")
clients_pd.head()

,first_name,last_name,age,gender,subscription
0,Findley,Burwin,43,Male,1
1,Ellette,Burwin,38,Female,1
2,Noe,Burwin,3,Male,1
3,Fabien,Keaves,46,Male,0
4,Forster,Culshew,19,Male,0


In [15]:
# count clients by last name to spot possible families
family_counts = (
    clients_pd
    .groupby("last_name")
    .size()
    .reset_index(name="client_count")
    .sort_values("client_count", ascending=False)
)

# add a binary flag for repeated surnames
family_lookup = family_counts.set_index("last_name")["client_count"] > 1
clients_pd["isFamily"] = clients_pd["last_name"].map(family_lookup).astype(int)

# show families
clients_pd[clients_pd["isFamily"] == 1].sort_values(["last_name", "age"], ascending=False)

,first_name,last_name,age,gender,subscription,isFamily
106,Reyna,Zannelli,82,Female,0,1
122,Quent,Zannelli,65,Male,0,1
103,Desirae,Zannelli,56,Female,0,1
96,Pauline,Zannelli,7,Female,1,1
99,Lolita,Zannelli,3,Female,1,1
182,Imogen,Scholl,97,Female,0,1
178,Aindrea,Scholl,55,Female,0,1
197,Stearne,Scholl,52,Male,0,1
185,Benedikt,Scholl,25,Male,0,1
180,Ernest,Scholl,24,Male,1,1


In [16]:
# clients go to swimming center

# from clients, select:
# - one family to go with every members
# - one family to go with one adult and one child
# - ten other people, to go independently if they have family or not

# sequence:

# 1. start one automatic with Nr 1001 and Banana-Cashes
    # cash={
    #             0.5: 200,
    #             1: 100,
    #             2: 100,
    #             5: 100,
    #             10: 50,
    #             50: 10,
    #         }

# 2. give money
    # give one member of the family {5:4, 10:2} or {10:3}
    # when the first member pay its ticket, the next member get the rest cash

    # give each alone person one of the options 
    # {5:1, 10:1}; {10:1}, {10:2}, {50:1}
    # but one of them has only {2:1}

    # families can have money to pay the ticket
    # one of the single visitors does not have enough money

# 3. Ticket option
    # families buy a LONG_TERM or DAY_PASS
    # alones buy randomic and select sauna randomic

    # families don't go to sauna
    # one third of single visitors go to sauna

# 4. after payment
    # the money goes back to the oldest family member.
    # alones have no changes, because they are alones.

# 5. after the execution, print a report from the automatic
#    - which tickets were sold
#    - show the rest cash

# 6. print a report of clients

# let's enjoy the swim day !

In [17]:
# select a simple example set for the swimming-center scenario

from random import randint, choice

family_names = list(clients_pd[clients_pd["isFamily"] == 1]["last_name"].unique())
print(f"existing family names: {family_names}")

# get a random family, where all goes together - - - 
one_family_name = choice(family_names)
family_pd1 = clients_pd[clients_pd["last_name"] == one_family_name]

family_names.remove(one_family_name)
one_family_name = choice(family_names)

# get a sample family, where only two members go - - -
family_pd2_adult = clients_pd[
    (clients_pd["last_name"] == one_family_name) & (clients_pd["age"] >= 18)
].sample(n=1)

family_pd2_child = clients_pd[
    (clients_pd["last_name"] == one_family_name) & (clients_pd["age"] < 18)
].sample(n=1)

# get 10 people, that go alone - - - 
solo_pd = clients_pd[
    (clients_pd["isFamily"] == 0) & (clients_pd["age"] >= 18)
].sample(n=10)

# show me!
visitors = pd.concat([family_pd1, family_pd2_adult, family_pd2_child, solo_pd], ignore_index=True) \
    .sort_values(["isFamily", "last_name", "age"], ascending=False)
visitors.head()

existing family names: ['Burwin', 'Mirralls', 'Northridge', 'Zannelli', 'Cheke', 'Scholl']


,first_name,last_name,age,gender,subscription,isFamily
5,Reyna,Zannelli,82,Female,0,1
6,Pauline,Zannelli,7,Female,1,1
2,Imogen,Scholl,97,Female,0,1
0,Aindrea,Scholl,55,Female,0,1
4,Stearne,Scholl,52,Male,0,1


In [18]:
# then give them money

from random import choice

family_cash_options = [{5: 4, 10: 2}, {10: 4}, {50: 1}]
solo_cash_options = [{5: 1, 10: 1}, {10: 1}, {10: 2}, {50: 1}]

visitors["cash"] = visitors["isFamily"].apply(
    lambda x: choice(family_cash_options) if x == 1 else choice(solo_cash_options)
)
visitors.loc[
    (visitors["isFamily"] == 1) & 
    (visitors["age"] != visitors.groupby("last_name")["age"].transform("max"))
, "cash"] = None

one_solo = visitors[visitors["isFamily"] == 0].sample(n=1)
idx = one_solo.index[0]
# idx
visitors.at[idx, "cash"] = {2: 1}

# show me!
visitors.head()

,first_name,last_name,age,gender,subscription,isFamily,cash
5,Reyna,Zannelli,82,Female,0,1,"{5: 4, 10: 2}"
6,Pauline,Zannelli,7,Female,1,1,None
2,Imogen,Scholl,97,Female,0,1,{10: 4}
0,Aindrea,Scholl,55,Female,0,1,None
4,Stearne,Scholl,52,Male,0,1,None


In [19]:
# then go to swimming center
from classes.enumerations import TicketOptions

# solo get ticktes randomic
visitors["ticket_option"] = visitors["isFamily"].apply(
    lambda x: None if x == 1 else choice(list(TicketOptions))
)

# families must get the same ticket for all members
family_ticket_options = [TicketOptions.DAY_PASS, TicketOptions.LONG_TERM]
names = list(visitors["last_name"].unique())
for n in names:
    ticket = choice(family_ticket_options)
    visitors.loc[visitors["last_name"] == n, "ticket_option"] = ticket

# sauna ?
visitors["sauna"] = visitors["isFamily"].apply(
    lambda x: 0 if x == 1 else randint(1,3)//3
)

# init ticket and client number
visitors["ticket_nr"] = 0
visitors["client_nr"] = visitors["subscription"].apply(
    lambda x: randint(1000, 1999) if x == 1 else 0
)

# force sort
# visitors = visitors.sort_values(["isFamily", "last_name", "age"], ascending=False)
# show me!
visitors

,first_name,last_name,age,gender,subscription,isFamily,cash,ticket_option,sauna,ticket_nr,client_nr
5,Reyna,Zannelli,82,Female,0,1,"{5: 4, 10: 2}",TicketOptions.DAY_PASS,0,0,0
6,Pauline,Zannelli,7,Female,1,1,None,TicketOptions.DAY_PASS,0,0,1574
2,Imogen,Scholl,97,Female,0,1,{10: 4},TicketOptions.LONG_TERM,0,0,0
0,Aindrea,Scholl,55,Female,0,1,None,TicketOptions.LONG_TERM,0,0,0
4,Stearne,Scholl,52,Male,0,1,None,TicketOptions.LONG_TERM,0,0,0
3,Benedikt,Scholl,25,Male,0,1,None,TicketOptions.LONG_TERM,0,0,0
1,Ernest,Scholl,24,Male,1,1,None,TicketOptions.LONG_TERM,0,0,1076
10,Alair,Treble,59,Male,0,0,{10: 1},TicketOptions.LONG_TERM,0,0,0
12,Issiah,Ranby,22,Male,0,0,{10: 2},TicketOptions.LONG_TERM,0,0,0
14,Traci,Potbury,85,Female,0,0,{10: 1},TicketOptions.LONG_TERM,0,0,0


In [20]:
# test cell - delete or comment it after use
# for idx in range(len(visitors)):
#     print(visitors.iloc[[idx]])
visitors


,first_name,last_name,age,gender,subscription,isFamily,cash,ticket_option,sauna,ticket_nr,client_nr
5,Reyna,Zannelli,82,Female,0,1,"{5: 4, 10: 2}",TicketOptions.DAY_PASS,0,0,0
6,Pauline,Zannelli,7,Female,1,1,None,TicketOptions.DAY_PASS,0,0,1574
2,Imogen,Scholl,97,Female,0,1,{10: 4},TicketOptions.LONG_TERM,0,0,0
0,Aindrea,Scholl,55,Female,0,1,None,TicketOptions.LONG_TERM,0,0,0
4,Stearne,Scholl,52,Male,0,1,None,TicketOptions.LONG_TERM,0,0,0
3,Benedikt,Scholl,25,Male,0,1,None,TicketOptions.LONG_TERM,0,0,0
1,Ernest,Scholl,24,Male,1,1,None,TicketOptions.LONG_TERM,0,0,1076
10,Alair,Treble,59,Male,0,0,{10: 1},TicketOptions.LONG_TERM,0,0,0
12,Issiah,Ranby,22,Male,0,0,{10: 2},TicketOptions.LONG_TERM,0,0,0
14,Traci,Potbury,85,Female,0,0,{10: 1},TicketOptions.LONG_TERM,0,0,0


In [21]:
# then here, we an run our example simulation. good !
from classes.automatics import Automatic
from classes.tickets import Ticket
from datetime import datetime

a1 = Automatic(1001, 
    cash={
        0.5: 200,
        1: 100,
        2: 100,
        5: 100,
        10: 50,
        50: 10,
    }
)

a1.report_cash()

ValueError: Automatic number 1001 already exists.

In [ ]:
# loop it

created_at = datetime.now().timestamp()
counter = 0
operation = None # for scope

while True:
    if visitors[visitors["ticket_nr"] == 0].empty:
        break
    if counter > len(visitors):
        raise TimeoutError("possible infinity loop.")

    idx = visitors[visitors["ticket_nr"] == 0].index[0]
    print(visitors.iloc[[idx]])
    
    created_at += randint(120, 3600) # the next comes between 2min e 1hr

    is_family = visitors.at[idx, "isFamily"]
    if is_family:
        # a family pay together and enter together
        family_name = visitors.at[idx, "last_name"]

        idx_cash = visitors[visitors["last_name"] == family_name]["cash"].dropna().index[0]
        idx0_cash = idx_cash
        cash = visitors.at[idx_cash, "cash"]

        # money = sum([k*v for k,v in cash.items()])

        idx_family = visitors[visitors["last_name"] == family_name].index.tolist()

        for x in idx_family:

            created_at += randint(60,300) # few minutes to buy between family members
            try:
                operation = a1.sell(
                    option = visitors.at[x, "ticket_option"],
                    cash = cash,
                    hasSubscription = visitors.at[x, "subscription"] == 1,
                    create_at = created_at,
                    allowSauna = visitors.at[x, "sauna"],
                    client = None if visitors.at[x, "subscription"] == 0 else visitors.at[x, "client_nr"]
                )
                cash = operation["change"]
                visitors.at[x, "cash"] = None
                visitors.at[x, "ticket_nr"] = operation["ticket"].ticket_number
            
                counter += 1
                print(f"{counter:3d}: created ticket nr {operation["ticket"].ticket_number} ")
            
            except Exception as e:
                print(f"Something went wrong: {e}")
                visitors.at[x, "ticket_nr"] = -1

        # then give the money back to the patriarch
        visitors.at[idx0_cash, "cash"] = cash


    else:
        # or one is not a family

        try:
            operation = a1.sell(
                option = visitors.at[idx, "ticket_option"],
                cash = visitors.at[idx, "cash"],
                hasSubscription = visitors.at[idx, "subscription"] == 1,
                create_at = created_at,
                allowSauna = visitors.at[idx, "sauna"],
                client = None if visitors.at[idx, "subscription"] == 0 else visitors["client_nr"]
            )
            visitors.at[idx, "cash"] = operation["change"]
            visitors.at[idx, "ticket_nr"] = operation["ticket"].ticket_number
        
            counter += 1
            print(f"{counter:3d}: created ticket nr {operation["ticket"].ticket_number} ")

        except Exception as e:
            print(f"Something went wrong: {e}")
            visitors.at[x, "ticket_nr"] = -1
        


In [ ]:
Ticket.report()

a1.report_cash()
            
